In [0]:
%python
# --- 
# PROJECT: NYC Ride Analytics
# STAGE: Gold Layer (Business Intelligence & Aggregation)
# OBJECTIVE: Generate high-level insights for executive reporting
# ---

# --- SECURITY CONFIGURATION (REDACTED FOR GITHUB) ---
# Professional Practice: Credentials should be managed via Secret Scopes or Key Vault.
# Example: client_secret = dbutils.secrets.get(scope="analytics-scope", key="adls-secret")
client_id = "REDACTED_CLIENT_ID"
tenant_id = "REDACTED_TENANT_ID"
client_secret = "REDACTED_CLIENT_SECRET"
storage_account_name = "datalakerides" 

# Setting session-level configurations for OAuth2 authentication with Azure AD (Entra ID)
# This provides the necessary permissions to read from the 'transformed-data' container
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

from pyspark.sql.functions import col, avg, sum, count, round

# --- DATA LOADING ---
# Defining the path for our cleaned Silver Delta table in ADLS Gen2
silver_path = f"abfss://transformed-data@{storage_account_name}.dfs.core.windows.net/ride_data_silver"

# Load the Silver data for final aggregation and KPI generation
df_silver = spark.read.format("delta").load(silver_path)

print("Status: Silver data loaded successfully. Ready for Gold aggregation (Credentials Redacted).")

Status: Silver data loaded. Ready for Gold aggregation.


In [0]:
%python
# ---
# AGGREGATION LOGIC:
# We are summarizing the performance of each taxi vendor.
# KPIs: Total Rides, Average Trip Distance, Average Tip, and Total Revenue.
# ---

df_gold_performance = df_silver.groupBy("vendor_id").agg(
    count("*").alias("total_rides"),
    round(avg("trip_distance"), 2).alias("avg_distance_miles"),
    round(avg("tip_amount"), 2).alias("avg_tip_amount"),
    round(sum("total_amount"), 2).alias("total_revenue_usd")
)

# Display the final business view
display(df_gold_performance)

vendor_id,total_rides,avg_distance_miles,avg_tip_amount,total_revenue_usd
CMT,23003,2.84,0.67,267067.38
VTS,24350,2.87,0.71,284826.43
DDS,2370,2.88,0.69,27995.59


In [0]:
%python
# ---
# SINK: Writing the final Gold table to the 'transformed-data' container
# This table is now ready for the dashboarding team.
# ---

gold_path = f"abfss://transformed-data@{storage_account_name}.dfs.core.windows.net/ride_performance_gold"

df_gold_performance.write.format("delta").mode("overwrite").save(gold_path)

print(f"MISSION ACCOMPLISHED: Gold insights are live at: {gold_path}")

MISSION ACCOMPLISHED: Gold insights are live at: abfss://transformed-data@datalakerides.dfs.core.windows.net/ride_performance_gold
